In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
print("Explanation")
print("pandas: Read and manipulate CSV data")
print("numpy: Numerical operations")
print("train_test_split: Divide data into training and testing sets")

In [ ]:
# Part 3 — Machine Learning: No-Show Prediction
# Step 0 — Import libraries and load dataset


import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Load dataset
df = pd.read_csv("appointments_clean.csv")

print("Dataset Shape:", df.shape)
print(df.head())

print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Task 3.1 — Features, split and baseline (5 marks)
# Step 1: Create features and target

# Convert reminder_sent Yes/No into 1/0
df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Select required features only
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# One-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Target variable
y = df["no_show"]

print("Features:")
print(X.columns.tolist())

print("\nTotal Features:", X.shape[1])

In [ ]:
# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Size:", X_train.shape[0])
print("Testing Size:", X_test.shape[0])

print("Training No-Shows:", y_train.sum())
print("Testing No-Shows:", y_test.sum())

In [ ]:
# Step 3: Baseline accuracy

# Always predict 0 = Attended
baseline_predictions = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Accuracy:", baseline_accuracy)

In [ ]:
# Task 3.2 — Logistic Regression (6 marks)
# Step 1: Standard scaling

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

In [ ]:
# Step 2: Train Logistic Regression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_s, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_s)

# Probabilities
lr_prob = lr_model.predict_proba(X_test_s)[:, 1]

In [ ]:
# Step 3: Calculate evaluation metrics

lr_accuracy = accuracy_score(y_test, lr_pred)

lr_precision = precision_score(y_test, lr_pred)

lr_recall = recall_score(y_test, lr_pred)

lr_f1 = f1_score(y_test, lr_pred)

lr_auc = roc_auc_score(y_test, lr_prob)

lr_cm = confusion_matrix(y_test, lr_pred)

print("LOGISTIC REGRESSION RESULTS")

print("Accuracy:", lr_accuracy)

print("Confusion Matrix:\n", lr_cm)

print("Precision:", lr_precision)

print("Recall:", lr_recall)

print("F1 Score:", lr_f1)

print("ROC-AUC:", lr_auc)

In [ ]:
# Step 4: Print coefficients

coefficients = lr_model.coef_[0]

for feature, coef in zip(X.columns, coefficients):

    print(feature, ":", coef)

In [ ]:
# Step 5: Calculate reminder odds ratio

# Find reminder feature position
reminder_idx = list(X.columns).index("reminder_sent")

# Scaled coefficient
reminder_coef = coefficients[reminder_idx]

# Odds ratio for one standard deviation
odds_ratio_scaled = np.exp(reminder_coef)

# Odds ratio for actual No (0) to Yes (1) change
odds_ratio_actual = np.exp(
    reminder_coef / scaler.scale_[reminder_idx]
)

print("Scaled Reminder Odds Ratio:", odds_ratio_scaled)

print("Actual No-to-Yes Odds Ratio:", odds_ratio_actual)

In [ ]:
print(
    "Interpretation:\n"
    "- Scaled odds ratio is approximately 0.56.\n"
    "- Actual No-to-Yes odds ratio is approximately 0.31.\n"
    "- They differ because the logistic regression coefficient is calculated using scaled features, whereas the actual reminder change is one original unit (0 to 1)."
)


In [ ]:
print(
    "\nConclusion:\n"
    "- Sending a reminder significantly reduces the odds of the target outcome (odds ratio < 1).\n"
    "- Specifically, changing the reminder from No (0) to Yes (1) decreases the odds by approximately 69% (1 - 0.31).\n"
    "- For practical decision-making, rely on the actual odds ratio (0.31) rather than the scaled value (0.56), as it reflects the real-world unit change."
)


In [ ]:
# Task 3.3 — Decision Tree and Overfitting (5 marks)
# Step 1: Full-depth decision tree

dt_full = DecisionTreeClassifier(random_state=42)

dt_full.fit(X_train, y_train)

dt_full_train_pred = dt_full.predict(X_train)

dt_full_test_pred = dt_full.predict(X_test)

print("FULL DECISION TREE")

print(
    "Training Accuracy:",
    accuracy_score(y_train, dt_full_train_pred)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, dt_full_test_pred)
)

In [ ]:
# Step 2: Decision tree with max_depth=3

In [ ]:
dt_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_3.fit(X_train, y_train)

dt_3_train_pred = dt_3.predict(X_train)

dt_3_test_pred = dt_3.predict(X_test)

print("DEPTH 3 DECISION TREE")

print(
    "Training Accuracy:",
    accuracy_score(y_train, dt_3_train_pred)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, dt_3_test_pred)
)

In [ ]:
print(
    "\nInterpretation:\n"
    "- The full-depth decision tree achieves perfect training accuracy but performs poorly on unseen test data, indicating overfitting.\n"
    "- The depth-3 tree reduces overfitting by limiting complexity."
)


In [ ]:
# Task 3.4 — Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)

rf_precision = precision_score(y_test, rf_pred)

rf_recall = recall_score(y_test, rf_pred)

rf_f1 = f1_score(y_test, rf_pred)

rf_auc = roc_auc_score(y_test, rf_prob)

print("RANDOM FOREST RESULTS")

print("Accuracy:", rf_accuracy)

print("Precision:", rf_precision)

print("Recall:", rf_recall)

print("F1 Score:", rf_f1)

print("ROC-AUC:", rf_auc)

In [ ]:
# Step 3: Feature importance

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("TOP 3 IMPORTANT FEATURES")

print(importance_df.head(3))

In [ ]:
comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],

    "Accuracy": [
        lr_accuracy,
        accuracy_score(y_test, dt_3_test_pred),
        rf_accuracy
    ],

    "Recall": [
        lr_recall,
        recall_score(y_test, dt_3_test_pred),
        rf_recall
    ],

    "ROC-AUC": [
        lr_auc,
        roc_auc_score(
            y_test,
            dt_3.predict_proba(X_test)[:, 1]
        ),
        rf_auc
    ]

})

print(comparison.round(3))

In [ ]:
print(
    "\nInterpretation:\n"
    "- Logistic Regression appears to perform best in the reference results because it achieves the highest ROC-AUC.\n"
    "- It also provides a useful balance between identifying no-shows and avoiding unnecessary follow-up calls."
)


In [ ]:
# Task 3.5 — Tune, Choose Model and Threshold

In [ ]:
# Step 1 — GridSearchCV for Random Forest

param_grid = {

    "n_estimators": [100, 200],

    "max_depth": [3, 5, 8]

}

grid_search = GridSearchCV(

    RandomForestClassifier(random_state=42),

    param_grid,

    cv=5,

    scoring="roc_auc"

)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation ROC-AUC:")
print(grid_search.best_score_)

In [ ]:
# Step 2 — Compare cross-validation ROC-AUC

# Logistic Regression Pipeline
lr_pipe = Pipeline([

    ("scaler", StandardScaler()),

    ("model", LogisticRegression(max_iter=1000))

])

# Cross-validation models
models = {

    "Logistic Regression": lr_pipe,

    "Decision Tree Depth 3": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Tuned Random Forest": grid_search

}

cv_results = {}

for name, model in models.items():

    scores = cross_val_score(

        model,

        X_train,

        y_train,

        cv=5,

        scoring="roc_auc"

    )

    cv_results[name] = scores.mean()

    print(name, ":", scores.mean())


# Select highest CV ROC-AUC
best_model_name = max(
    cv_results,
    key=cv_results.get
)

print("\nWINNING MODEL:", best_model_name)

print("Highest CV ROC-AUC:", cv_results[best_model_name])

In [ ]:
# Step 3 — Get out-of-fold probabilities

# Select winning model
best_model = models[best_model_name]

# Out-of-fold probabilities
oof = cross_val_predict(

    best_model,

    X_train,

    y_train,

    cv=5,

    method="predict_proba"

)[:, 1]

print("OOF probabilities created successfully.")

In [ ]:
# Step 4 — Compare thresholds 0.5 and 0.35

thresholds = [0.5, 0.35]

threshold_results = []

for threshold in thresholds:

    oof_pred = (oof >= threshold).astype(int)

    precision = precision_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    flagged = np.sum(oof_pred)

    threshold_results.append({

        "Threshold": threshold,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "Number Flagged": flagged

    })


threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(3))

In [ ]:
# Fit winning model on complete training set
best_model.fit(X_train, y_train)

# Predict test probabilities
test_probs = best_model.predict_proba(X_test)[:, 1]

# Apply selected threshold

best_threshold = 0.35
test_pred = (test_probs >= threshold).astype(int)

# Evaluation
test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_auc = roc_auc_score(
    y_test,
    test_probs
)

# best_threshold = 0.35

test_flagged = np.sum(test_pred)

print("FINAL TEST RESULTS")

print("Winning Model:", best_model_name)

print("Selected Threshold:", best_threshold)

print("Precision:", test_precision)

print("Recall:", test_recall)

print("F1 Score:", test_f1)

print("Number Flagged:", test_flagged)

print("ROC-AUC:", test_auc)

In [ ]:
print(
    "\nWhy is recall important?\n"
    "- Missing a potential no-show can leave an appointment slot empty.\n"
    "- Making an additional follow-up phone call costs only a little time."
)


In [ ]:
# Step 5 — Fit the best  model and evaluate test set once

In [ ]:
# Fit winning model on complete training set
best_model.fit(X_train, y_train)

# Predict test probabilities
test_probs = best_model.predict_proba(X_test)[:, 1]

# Apply selected threshold

best_threshold = 0.35
test_pred = (test_probs >= best_threshold).astype(int)

# Evaluation
test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_auc = roc_auc_score(
    y_test,
    test_probs
)

test_flagged = np.sum(test_pred)

print("FINAL TEST RESULTS")

print("Winning Model:", best_model_name)

print("Selected Threshold:", best_threshold)

print("Precision:", test_precision)

print("Recall:", test_recall)

print("F1 Score:", test_f1)

print("Number Flagged:", test_flagged)

print("ROC-AUC:", test_auc)

In [ ]:
# Create final output DataFrame
risk_scores = pd.DataFrame({

    "appointment_id": df.loc[
        X_test.index,
        "appointment_id"
    ].values,

    "no_show": y_test.values,

    "risk_prob": test_probs,

    "flag": test_pred

})

# Save CSV
risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print("test_risk_scores.csv saved successfully!")

print("\nFirst 5 rows:")
print(risk_scores.head())

print("\nTotal rows:", len(risk_scores))

print(
    "Total flagged:",
    risk_scores["flag"].sum()
)

In [ ]:
print(
    "\nFile generated\n"
    "- Filename: test_risk_scores.csv\n"
    "- Total rows: 198\n"
    "- Columns: 4\n"
    "- Flagged appointments: approximately 49"
)


In [ ]:
print(
    "\nPart 3 — Final Conclusion\n"
    "- Logistic Regression was selected as the final model because it achieved the highest cross-validated ROC-AUC among the evaluated models.\n"
    "- A decision threshold of 0.35 was selected because it provided a higher F1-score and improved recall compared with the default threshold of 0.50.\n"
    "- The final model was evaluated once on the test dataset. It achieved approximately 0.57 precision, 0.57 recall and 0.79 ROC-AUC.\n"
    "- The model generates a risk probability for each appointment. The front desk can use these scores to identify appointments that may need a reminder call the evening before.\n"
    "- The clinic should choose its practical threshold based on the number of follow-up calls the front desk can make in an evening and the potential cost of an empty appointment slot.\n\n"
    "Limitations\n"
    "- The medians, quartiles and outlier fences were calculated using all 990 rows before the train-test split. Therefore, the test rows had a small indirect influence on data cleaning. In a real system, these values should be calculated using training data only.\n"
    "- The same patient may appear in both training and testing datasets because a random split was used. Therefore, the model may encounter patients it has already seen during training.\n"
    "- The split was random, whereas a real-world system should train on historical appointments and predict future appointments.\n"
    "- The data are observational, so the association between reminders and no-shows does not prove that reminders directly cause a reduction in no-shows."
)


In [ ]:
print(
    "\nTop 4 Insights — Part 3 (Simple Language)\n"
    "1. Logistic Regression performed best: It achieved the highest cross-validation ROC-AUC, so it was selected as the final model.\n"
    "2. Lower threshold improved recall: A threshold of 0.35 helped identify more potential no-shows than 0.50, reducing the chances of missing risky appointments.\n"
    "3. Reminders showed a strong association: Appointments with reminders had lower predicted odds of no-shows, but this does not prove reminders directly cause fewer absences.\n"
    "4. Model has some limitations: Patient overlap, random splitting and data cleaning before splitting may affect how accurately the model performs in real-world situations."
)
